# AMADEUS: Adaptive Music & Audio Diffusion Engine
### Accelerated Google Colab GPU Training & Inference Notebook

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Active GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

## 2. Install Required Dependencies & Configure Environment

In [ ]:
!pip install -q torchaudio soundfile librosa transformers accelerate tqdm fastapi uvicorn pydantic pyyaml websockets
!pip install -e /content/AMADEUS 2>/dev/null || !pip install -e .
import sys
import os
for p in ['/content/AMADEUS', '/content/amadeus', '.']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
if os.path.exists('/content/AMADEUS') and not os.path.exists('/content/amadeus'):
    try:
        os.symlink('/content/AMADEUS', '/content/amadeus')
    except Exception:
        pass
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

## 3. (Optional) Mount Google Drive for Checkpoint Persistence

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/amadeus_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints directory set to:', CHECKPOINT_DIR)

## 4. Prepare Music Audio Dataset

In [ ]:
import sys
import os
for p in ['/content/AMADEUS', '/content/amadeus', '.']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
if os.path.exists('/content/AMADEUS') and not os.path.exists('/content/amadeus'):
    try:
        os.symlink('/content/AMADEUS', '/content/amadeus')
    except Exception:
        pass
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
from amadeus.data.dataset import download_demo_fma_dataset, MusicAudioDataset
DATASET_DIR = './fma_dataset'
download_demo_fma_dataset(DATASET_DIR)
ds = MusicAudioDataset(DATASET_DIR, duration_seconds=4.0)
print('Dataset ready with sample count:', len(ds))

## 5. Train Stage 1: Neural Codec VAE (AMP + Multi-Period Discriminator)

In [ ]:
from amadeus.scripts.train_codec import train_codec
train_codec(
    data_dir=DATASET_DIR,
    epochs=2,
    batch_size=4,
    lr=2e-4,
    save_dir='./checkpoints',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    grad_accum=2
)

## 6. Train Stage 2: Flow Matching Diffusion Transformer (SDPA + AMP)

In [ ]:
from amadeus.scripts.train_dit import train_dit
train_dit(
    data_dir=DATASET_DIR,
    epochs=2,
    batch_size=4,
    lr=1e-4,
    save_dir='./checkpoints',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    grad_accum=2
)

## 7. Train Stage 3: Semantic Token LM (KV Caching + SDPA)

In [ ]:
from amadeus.scripts.train_lm import train_lm
train_lm(
    data_dir=DATASET_DIR,
    epochs=2,
    batch_size=4,
    lr=1.5e-4,
    save_dir='./checkpoints',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    grad_accum=2
)

## 8. Run Evaluation Suite & Metrics

In [ ]:
from amadeus.pipeline.inference import AMADEUSPipeline
from amadeus.eval.metrics import EvaluationSuite

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipeline = AMADEUSPipeline(device=device, checkpoint_dir='./checkpoints')
suite = EvaluationSuite()
metrics = suite.run_full_evaluation(pipeline, test_samples=2)
print('Mean Latency (ms):', metrics['mean_latency_ms'])
print('Mean STFT Distance:', metrics['mean_stft_distance'])
print('Mean Style Cosine:', metrics['mean_style_cosine'])
print('FAD Score:', metrics['fad_score'])
print('Beat F1:', metrics['mean_beat_f1'])
print('Targets Status:', metrics['targets_met'])

## 9. Interactive Generation & In-Browser Audio Playback

In [ ]:
import IPython.display as ipd

prompt = 'lo-fi chillhop beat with soft jazz piano and vinyl warmth'
print('Generating track for prompt:', prompt)
result = pipeline.generate(
    text_prompt=prompt,
    controls={'steps': 8, 'duration_s': 6, 'stems': True}
)

audio_np = result['audio'].cpu().numpy()
print('Job ID:', result['job_id'])
print('Watermark ID:', result['watermark_id'])
print('Key:', result['analysis']['key'], result['analysis']['mode'], '| BPM:', result['analysis']['bpm'])

ipd.display(ipd.Audio(audio_np, rate=48000))